# OpenAI API

In [13]:
%pip install -q openai pandas pydantic python-dotenv

ERROR: Could not find a version that satisfies the requirement openai_call_metrics (from versions: none)
ERROR: No matching distribution found for openai_call_metrics


In [22]:
import asyncio
import html
import os
import re
import time
from typing import Literal, Optional
from google.colab import userdata

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import BaseModel, Field

from openai_call_metrics import (
    OpenAIMetricsSession,
    achat_completions_parse_with_metrics,
    chat_completions_create_with_metrics,
    chat_completions_parse_with_metrics,
)

# Dane

In [12]:
import zipfile
from pathlib import Path

# Nazwa pliku ZIP przesłanego do JupyterLab
zip_path = Path("real_estate_ads.csv(1).zip")

# Folder, do którego rozpakujemy archiwum
output_dir = Path("data")

output_dir.mkdir(parents=True, exist_ok=True)

# Rozpakowanie
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(output_dir)

print(f"Rozpakowano do: {output_dir.resolve()}")

# Wyświetlenie plików
for path in output_dir.rglob("*"):
    if path.is_file():
        print(f"{path}: {path.stat().st_size / 1024**2:.1f} MB")

Rozpakowano do: /content/data
data/real_estate_ads.csv: 36.2 MB
data/__MACOSX/._real_estate_ads.csv: 0.0 MB


In [14]:
plik = pd.read_csv("data/real_estate_ads.csv")
plik.head()

,id,market,created_at_first,created_at,district_lon,district_lat,title,description,params,price,no_rooms,m,price_per_m,map_lon,map_lat
0,49341,secondary,2018-01-04 13:11:05,2018-01-08 11:27:35,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,3,48.000000,6020.83,16.959516,52.391494
1,261720,secondary,2018-01-15 19:12:07,2018-01-25 11:11:31,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>379000<br>price[curre...,379000.0,3,61.000000,6213.11,16.950743,52.380535
2,50441,secondary,2018-01-04 13:53:29,2018-01-11 05:55:12,16.9408,52.38356,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,3,63.000000,5380.95,16.945860,52.393188
3,203518,secondary,2018-01-12 11:10:57,2018-01-30 14:38:39,16.9408,52.38356,"Mieszkanie 3 pokojowe blisko tramwaju, Poznań os.",<p>Oferuję na sprzedaż jasne i zadbane mieszka...,price<=>price<br>price<=>330000<br>price[curre...,330000.0,3,59.200001,5574.32,16.980279,52.391129
4,5,secondary,2018-01-02 11:34:51,2018-01-18 14:22:29,16.9408,52.38356,"Inwestycja -rondo Rataje student AWF,POLITECHNIKA",<p><strong>Dwa pokoje na niskim piętrze tuż pr...,price<=>price<br>price<=>259000<br>price[curre...,259000.0,2,38.000000,6815.79,16.950027,52.395861


Klucz

In [15]:
# Wczytuje OPENAI_API_KEY z pliku .env w folderze projektu.
# .env jest w .gitignore - nigdy nie commituj swojego klucza.
load_dotenv()

api_key = userdata.get("OPENAI-API-KEY")
if not api_key:
    raise ValueError(
        "Nie znaleziono OPENAI_API_KEY. Zrób jedno z dwóch: "
        "(1) Utwórz plik .env obok tego notebooka z treścią: OPENAI_API_KEY=sk-... "
        "(2) Albo wyeksportuj klucz przed uruchomieniem Jupytera: export OPENAI_API_KEY=sk-... "
        "Notebooki w Cursor/VSCode często nie dziedziczą zmiennych wyeksportowanych w terminalu."
    )

client = OpenAI(api_key=api_key, max_retries=5)  # SDK ponawia zapytania przy limitach / chwilowych błędach

## Ustawienia, których używamy

Porównujemy dwa modele (ceny za 1M tokenów wejściowych / wyjściowych, sprawdzone 2026-10-23):

| Model | Cena | Do czego |
|---|---|---|
| `gpt-6-luna` | $0.10 / $0.50 | najnowszy tani model do dużej liczby prostych zadań |
| `gpt-5.6-luna` | $0.20 / $1.20 | poprzednia generacja, ok. 2× droższa |

- **`reasoning_effort="none"`**: nowe modele przed odpowiedzią „myślą” w ukrytych tokenach, za które płacimy. Odczytanie faktów z ogłoszenia nie wymaga myślenia, więc to wyłączamy. Oba modele mają te same ustawienia, więc porównujemy same modele.
- **`max_completion_tokens=1000`**: górny limit długości odpowiedzi, zabezpieczenie przed zbyt długą (i drogą) odpowiedzią.
- **`temperature` nie ustawiamy**: modele rozumujące zwykle akceptują tylko wartość domyślną i zwracają błąd 400 przy innej.

In [16]:
MODELS = ["gpt-6-luna", "gpt-5.6-luna"]
MAX_COMPLETION_TOKENS = 1_000

## 1. Wczytanie i czyszczenie danych

- Case dotyczy **rynku wtórnego**, więc filtrujemy po kolumnie `market`.
- **Od razu dzielimy dane na zbiór treningowy i testowy.
- Opisy są w HTML. Usunięcie tagów oszczędza tokeny (= pieniądze).

In [19]:
df = plik.loc[plik['market'].eq('secondary')].copy()
df['published_at'] = pd.to_datetime(df['created_at_first'], errors='coerce')
df['updated_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
valid_target = np.isfinite(df['price']) & df['price'].gt(0)
print('Wykluczone: brak daty lub poprawnej ceny:',
      int((~valid_target | df['published_at'].isna()).sum()))
df = (df.loc[valid_target & df['published_at'].notna()]
      .sort_values(['published_at', 'updated_at'], kind='stable')
      .drop_duplicates('id', keep='first').copy())
if len(df) < 2:
    raise ValueError('Za mało ofert do podziału.')
cutoff = df['published_at'].iloc[int(0.8 * len(df))].normalize()
train_idx = df.index[df['published_at'] < cutoff]
test_idx = df.index[df['published_at'] >= cutoff]
if not len(train_idx) or not len(test_idx):
    raise ValueError('Granica dzienna daje pusty trening lub test.')
assert set(train_idx).isdisjoint(test_idx)
assert set(df.loc[train_idx, 'id']).isdisjoint(df.loc[test_idx, 'id'])
assert df.loc[train_idx, 'published_at'].max() < df.loc[test_idx, 'published_at'].min()
print('Granica testu:', cutoff.date())
print('Trening:', len(train_idx), '| test:', len(test_idx))

df['m'] = pd.to_numeric(df['m'], errors='coerce')
df['m'] = df['m'].where(np.isfinite(df['m']) & df['m'].gt(0))
coords = df[['map_lat', 'map_lon', 'district_lat', 'district_lon']].apply(pd.to_numeric, errors='coerce')
coords_ok = (coords['map_lat'].between(-90, 90) & coords['district_lat'].between(-90, 90)
             & coords['map_lon'].between(-180, 180) & coords['district_lon'].between(-180, 180))
lat1, lon1, lat2, lon2 = [np.radians(coords[c]) for c in coords.columns]
a = np.sin((lat1-lat2)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon1-lon2)/2)**2
df['dist_from_c'] = (6371 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))).where(coords_ok)
df['no_rooms'] = df['no_rooms'].fillna('unknown').astype(str)

Wykluczone: brak daty lub poprawnej ceny: 0
Granica testu: 2020-05-24
Trening: 4758 | test: 1196


In [31]:
def clean_description(raw: str, max_chars: int = 4000) -> str:
    """Zamienia opis w HTML na zwykły tekst (i przycina bardzo długie opisy)."""
    text = html.unescape(str(raw))
    text = re.sub(r"<br\s*/?>|</p>|</li>", "\n", text)  # zachowujemy podział na linie
    text = re.sub(r"<[^>]+>", " ", text)                 # usuwamy pozostałe tagi
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()[:max_chars]

df['params_clean'] = df['params'].map(clean_description)
TEXT_VARS = ['title', 'description', 'params_clean']
df[TEXT_VARS] = df[TEXT_VARS].fillna('').astype(str)
df_train = df.loc[train_idx].copy()
df_test = df.loc[test_idx].copy()
y_train, y_test = df_train['price'], df_test['price']
assert not df['params_clean'].str.contains(r'(?:price|price_per_m)(?:\[[^]]*\])?<=>', regex=True).any()

sample = df_train.sample(n=5, random_state=42)
print(f"{len(plik)} ogłoszeń łącznie, {len(df)} z rynku wtórnego: "
      f"{len(df_train)} train / {len(df_test)} test, {len(sample)} w próbce do porównania")

print(clean_description(sample["description"].iloc[0]))

13745 ogłoszeń łącznie, 5954 z rynku wtórnego: 4758 train / 1196 test, 5 w próbce do porównania
Witam,
 Posiadam na sprzedaż, bądź zamianę mieszkanie o powierzchni użytkowej 74m2.
 Lokal znajduję się na ostatnim piętrze 4-piętrowego bloku w cichej, zielonej okolicy. (os. Stare Żegrze)
 Mieszkanie składa się z czterech przestronnych pokoi, dwóch korytarzy, kuchni oraz łazienki z osobnym WC. 
 Pierwszy ukazuję się naszym oczom mniejszy korytarz, zarówno po jego lewej jak i prawej stronie znajdują się ok.10m pokoje. Idąc dalej dochodzimy do dużego korytarza z szafą typu komandor. Po lewej stronie znajduje się duży salon ok. 22m z balkonem. Zaś po prawej stronie mamy kolejny pokój o wielkości ok. 9m.
 W dalszej części korytarza znajduje się łazienka ok. 3,5m oraz osobna toaleta o wielkości ok. 1,5m. 
 Na samym końcu dochodzimy do prawie 7 metrowej kuchni. 
 Wszystkie pomieszczenia są nieprzechodnie, co wpływa na ich świetną ustawność. 
 Do mieszkania przynależy piwnica.
 Miesięczny czynsz 

## 2. Pierwsze wywołanie: co jest w ogłoszeniach?

Zwykłe zapytanie z odpowiedzią tekstową. Dajemy modelowi 20 ogłoszeń **ze zbioru treningowego** i listę kolumn, które już mamy, i prosimy o propozycję **nowych** cech. Odpowiedź czytamy sami i na jej podstawie wybieramy cechy do schematu w kroku 3.

Wiadomość ma dwie role:
- `system`: instrukcja, jak model ma się zachować,
- `user`: dane, na których ma pracować.

In [32]:
# Kolumny, które już mamy: klucze z `params` plus główne kolumny zbioru danych.
params_keys = sorted({
    kv.split("<=>")[0]
    for p in df_train["params"].dropna()
    for kv in p.split("<br>") if "<=>" in kv
})
known_columns = ["m", "no_rooms", "price", "map_lat", "map_lon"] + params_keys

ads_text = "\n\n".join(
    f"--- OGŁOSZENIE {i + 1} ---\n{clean_description(d, max_chars=1500)}"
    for i, d in enumerate(df_train.drop(sample.index).sample(n=20, random_state=0)["description"])  # tylko train!
)

explore_metrics = OpenAIMetricsSession()
response = chat_completions_create_with_metrics(
    client,
    session=explore_metrics,
    model="gpt-6-luna",
    messages=[
        {
            "role": "system",
            "content": (
                "You help build a model that predicts the price per m2 of apartments in Poznań. "
                "Read the ads below and propose 10-15 features that can be read from the description text, "
                "are likely to affect the price per m2 and appear in many ads. "
                f"Do NOT propose features already available as columns: {', '.join(known_columns)}. "
                "For each feature give an English snake_case name and one sentence in Polish explaining it. "
                "Answer in Polish."
            ),
        },
        {"role": "user", "content": ads_text},
    ],
    reasoning_effort="low",  # tu model ma porównać 20 ogłoszeń, więc pozwalamy mu trochę pomyśleć
    max_completion_tokens=4_000,
)
print(response.choices[0].message.content)
print()
print(explore_metrics.format_summary())

1. `neighborhood_name` — Nazwa osiedla lub części Poznania podana w ogłoszeniu pozwala uchwycić lokalne różnice cenowe, których nie opisują same współrzędne.
2. `transit_accessibility` — Informacje o bliskości przystanków tramwajowych i autobusowych oraz jakości dojazdu komunikacją miejską wskazują na wygodę codziennych podróży.
3. `city_center_travel_time` — Deklarowany czas dojazdu do centrum może odzwierciedlać atrakcyjność lokalizacji dla osób dojeżdżających.
4. `nearby_amenities` — Obecność w pobliżu sklepów, szkół, przedszkoli, przychodni czy centrów handlowych zwiększa wygodę życia i może podnosić wartość mieszkania.
5. `nearby_green_areas` — Wzmianki o parkach, terenach zielonych, spacerowych lub rekreacyjnych opisują walory otoczenia.
6. `balcony_or_loggia_area` — Podana w ogłoszeniu powierzchnia balkonu lub loggii pozwala odróżnić niewielki balkon od większej przestrzeni użytkowej.
7. `renovation_recency` — Informacja o roku lub czasie od ostatniego remontu przybliża aktualno

## 3. Schemat: jakie cechy chcemy dostać

Odpowiedź tekstowa za każdym razem wygląda trochę inaczej, a do modelu ceny potrzebujemy **tych samych kolumn dla każdego ogłoszenia**. Dlatego opisujemy cechy jako klasę Pydantic i przekazujemy ją do API (Structured Outputs). Model musi wtedy zwrócić dokładnie taki format.

- `Literal[...]`: model musi wybrać jedną z podanych wartości (cecha kategoryczna).
- `Optional[...]`: model może zwrócić `None`, gdy ogłoszenie o czymś nie wspomina. Chcemy *braku danych*, a nie zgadywania.
- `description=`: trafia do modelu, więc działa jak instrukcja (dlatego po angielsku).

**Wybrane cechy**

*   neighborhood_name
*   near_public_transport
*   city_center_travel_time
*   nearby_green_areas
*   renovation_recendly
*   finish_standard
*   layout_functionality
*   natural_light_exposure
*   balcony_area_sqm
*   num_bathrooms
*   has_elevators
*   has_storage



In [33]:
class ApartmentFeatures(BaseModel):
    neighborhood_name: Optional[str] = Field(description="Name of the neighborhood, if mentioned.")
    near_public_transport: Optional[bool] = Field(description="Tram/bus stops mentioned as nearby.")
    city_center_travel_time: Optional[float] = Field(
        description="Declared travel time to the city center in minutes."
    )
    nearby_green_areas: Optional[str] = Field(
        description="Mentions of parks, green spaces, walking paths, or recreational areas around the property."
    )
    renovation_recency: Optional[str] = Field(
        description="Information about the year or time elapsed since the last renovation, indicating the freshness of installations and finish."
    )
    finish_standard: Optional[Literal["shell_and_core", "standard", "high_standard", "luxury"]]= Field(
        description="Declared interior finish standard, from developer state to high-standard and luxury."
    )
    layout_functionality: Optional[Literal["separate", "walk_through", "open_kitchen"]] = Field(
        description="Whether the rooms are separate, walk-through or connected to kitchen."
    )
    balcony_area_sqm: Optional[float] = Field(
        description="Area of the balcony, loggia, or terrace in square meters, if mentioned in the ad."
    )
    natural_light_exposure: Optional[str] = Field(
        description="Descriptions of sunlight exposure, apartment brightness, or window orientation directions affecting comfort."
    )
    num_bathrooms: Optional[int] = Field(description="Number of bathrooms, if stated.")
    parking: Optional[Literal["none", "street", "parking_spot", "garage"]] = Field(
        description="Best parking option included or mentioned for the apartment."
    )
    has_elevator: Optional[bool] = Field(description="Building has an elevator (winda).")
    has_storage: Optional[bool] = Field(description="Basement or storage room (piwnica, komórka lokatorska).")

## 4. Ekstrakcja tymi samymi ogłoszeniami w obu modelach

Zamiast `create` używamy `client.chat.completions.parse(...)` z `response_format=ApartmentFeatures`. Gotowy obiekt jest w `message.parsed`, nie trzeba samemu parsować tekstu. Każdy model ma osobny licznik kosztów, żeby łatwo je porównać.

In [34]:
SYSTEM_PROMPT = """You are a data extraction assistant for real estate listings.
You receive a Polish apartment-for-sale ad from Poznań (secondary market).
Extract the requested fields using ONLY information stated in the ad.
If something is not mentioned, return null - never guess."""


def build_messages(description: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": clean_description(description)},
    ]


def extract_features(description: str, model: str, metrics: OpenAIMetricsSession) -> ApartmentFeatures:
    response = chat_completions_parse_with_metrics(
        client,
        session=metrics,
        model=model,
        messages=build_messages(description),
        response_format=ApartmentFeatures,
        reasoning_effort="none",
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    return response.choices[0].message.parsed

In [35]:
# Zabezpieczenie przed przypadkowym uruchomieniem porównania na dużej liczbie ogłoszeń.
N_ADS = 10
assert N_ADS <= 100, "Porównanie modeli rób na maks. 100 ogłoszeniach; cały zbiór przetwarzasz w kroku 6."

results = {}  # model -> DataFrame z cechami
metrics = {}  # model -> licznik kosztów i czasu

for model in MODELS:
    metrics[model] = OpenAIMetricsSession()
    rows = []
    for ad_id, description in zip(sample["id"], sample["description"]):
        try:
            features = extract_features(description, model, metrics[model])
        except Exception as e:  # jedno problematyczne ogłoszenie nie powinno zatrzymać całej pętli
            print(f"{model} id={ad_id} błąd: {e}")
            continue
        rows.append({"id": ad_id, **features.model_dump()})
    results[model] = pd.DataFrame(rows).set_index("id")
    print(f"{model}: gotowe, {len(rows)} ogłoszeń")

results[MODELS[0]]

gpt-6-luna: gotowe, 5 ogłoszeń
gpt-5.6-luna: gotowe, 5 ogłoszeń


,neighborhood_name,near_public_transport,city_center_travel_time,nearby_green_areas,renovation_recency,finish_standard,layout_functionality,balcony_area_sqm,natural_light_exposure,num_bathrooms,parking,has_elevator,has_storage
id,,,,,,,,,,,,,
6098869,os. Stare Żegrze,True,None,"Cicha, zielona okolica; place zabaw, siłownia ...",Mieszkanie do odświeżenia.,None,separate,None,None,1,None,None,True
925360,Rataje,True,None,None,None,shell_and_core,open_kitchen,None,None,1,parking_spot,None,True
6139961,centrum miasta,True,None,"Warta, park, tereny spacerowe Poznańskiej Malty",None,high_standard,open_kitchen,None,Widok na rzekę Wartę i park,2,garage,None,True
1346365,os. Orła Białego,True,None,None,None,None,separate,None,Pokoje: południowy wschód; kuchnia: północny z...,1,street,None,True
1453765,None,True,None,"Liczne tereny zielone, spacerowe i rekreacyjne...","Mieszkanie będzie wykończone; nowe instalacje,...",high_standard,separate,None,None,1,None,None,True


## 5. Porównanie modeli

### Zgodność odpowiedzi
Jak często oba modele dały tę samą odpowiedź (brak danych w obu też liczymy jako zgodność)?

In [36]:
model_a, model_b = MODELS
a, b = results[model_a], results[model_b]
common = a.index.intersection(b.index)
a, b = a.loc[common], b.loc[common]

agreement = ((a == b) | (a.isna() & b.isna())).mean().round(2)
agreement.rename("zgodność")

,zgodność
neighborhood_name,0.8
near_public_transport,1.0
city_center_travel_time,1.0
nearby_green_areas,0.4
renovation_recency,0.4
finish_standard,0.8
layout_functionality,1.0
balcony_area_sqm,1.0
natural_light_exposure,0.4
num_bathrooms,1.0


Zgodności dla większości cech są wysokie. Dla cech natural_light_exposure, nearby_green_areas,
renovation_recency można się zastanowić nad jasnością opisu. Dla pozostałych cech modele bardzo zgodne.

In [37]:
# Tylko ogłoszenia i cechy, w których modele się różnią.
a.compare(b, result_names=(model_a, model_b))

neighborhood_name               \
               gpt-6-luna gpt-5.6-luna   
id                                       
6098869               NaN          NaN   
6139961    centrum miasta         None   
1346365               NaN          NaN   
1453765               NaN          NaN   

                                        nearby_green_areas  \
                                                gpt-6-luna   
id                                                           
6098869  Cicha, zielona okolica; place zabaw, siłownia ...   
6139961    Warta, park, tereny spacerowe Poznańskiej Malty   
1346365                                                NaN   
1453765  Liczne tereny zielone, spacerowe i rekreacyjne...   

                                                            \
                                              gpt-5.6-luna   
id                                                           
6098869  Cicha, zielona okolica; place zabaw, siłownia ...   
6139961  rzeka Warta, park, tereny spacerowe Poznańskie...   
1346365                                                NaN   
1453765  Liczne tereny zielone, spacerowe oraz tereny r...   

                                        renovation_recency  \
                                                gpt-6-luna   
id                                                           
6098869                         Mieszkanie do odświeżenia.   
6139961                                                NaN   
1346365                                               None   
1453765  Mieszkanie będzie wykończone; nowe instalacje,...   

                                                           finish_standard  \
                                              gpt-5.6-luna      gpt-6-luna   
id                                                                           
6098869                                     Do odświeżenia             NaN   
6139961                                                NaN   high_standard   
1346365  W ubiegłym roku ocieplono blok i położono nową...             NaN   
1453765  Mieszkanie będzie wykończone; nowe instalacje,...             NaN   

                                                 natural_light_exposure  \
        gpt-5.6-luna                                         gpt-6-luna   
id                                                                        
6098869          NaN                                                NaN   
6139961       luxury                        Widok na rzekę Wartę i park   
1346365          NaN  Pokoje: południowy wschód; kuchnia: północny z...   
1453765          NaN                                               None   

                                                           has_storage  \
                                              gpt-5.6-luna  gpt-6-luna   
id                                                                       
6098869                                                NaN         NaN   
6139961                        widok na rzekę Wartę i park        True   
1346365  Pokoje: południowy wschód; kuchnia: północny z...         NaN   
1453765                         Przepiękny widok na zieleń         NaN   

                      
        gpt-5.6-luna  
id                    
6098869          NaN  
6139961         None  
1346365          NaN  
1453765          NaN

### Koszt i czas
Przy `reasoning_effort="none"` kolumna `reasoning` powinna mieć 0. Ostatnia kolumna szacuje koszt dla **wszystkich** ogłoszeń z rynku wtórnego.

In [40]:
summary = []
for model, m in metrics.items():
    n = len(m.calls)
    summary.append({
        "model": model,
        "ogłoszeń": n,
        "śr. tokeny in": round(m.total_input_tokens / n),
        "śr. tokeny out": round(m.total_output_tokens / n),
        "śr. reasoning": round(sum(c.reasoning_tokens for c in m.calls) / n),
        "śr. czas [s]": round(m.total_elapsed_s / n, 2),
        "koszt [$]": round(m.total_cost_usd, 4),
        f"cały rynek wtórny ({len(df)}) [$]": round(m.total_cost_usd / n * len(df), 2),
    })
pd.DataFrame(summary).set_index("model")

,ogłoszeń,śr. tokeny in,śr. tokeny out,śr. reasoning,śr. czas [s],koszt [$],cały rynek wtórny (5954) [$]
model,,,,,,,
gpt-6-luna,5,1209,126,0,2.70,0.0009,1.10
gpt-5.6-luna,5,1209,131,0,2.24,0.0020,2.37


Modele radzą sobie różnie, dla niektórych cech gpt-6-luna lepiej wyłapuje informacje, dla innych gpt-5.6-luna. Ze względu na to, przeważającym czynnikiem są koszty- wybrany model to gpt-6-luna.

## 6. Cały zbiór wybranym modelem (asyncio)

Dla wybranego modelu gpt-6-luna przetwarzamy wszystkie ogłoszenia z rynku wtórnego, treningowe i testowe. To nie jest wyciek danych: ekstrakcja odczytuje fakty z każdego ogłoszenia osobno i nie korzysta z ceny.

Wysyłanie zapytań po kolei trwałoby 2–3 godziny, bo prawie cały czas czekamy na odpowiedź serwera. Z `asyncio` wysyłamy wiele zapytań naraz:
- `AsyncOpenAI` to asynchroniczna wersja klienta, a jego metody wywołujemy z `await`,
- `asyncio.Semaphore(CONCURRENCY)` pilnuje, żeby naraz leciało maksymalnie `CONCURRENCY` zapytań (limit dla `gpt-6-luna` to 5000 zapytań/min),
- `asyncio.gather(...)` uruchamia wszystkie zadania z paczki i czeka, aż się skończą,
- w Jupyterze piszemy po prostu `await ...` w komórce (`asyncio.run()` w notebooku nie działa).

Wyniki zapisujemy do `llm_features.csv` co `CHUNK_SIZE` ogłoszeń. Jeśli coś przerwie działanie, ponowne uruchomienie pominie gotowe ogłoszenia, więc nie płacisz dwa razy.

In [41]:
SELECTED_MODEL = "gpt-6-luna"  # model wybrany na podstawie porównania z kroku 5
CONCURRENCY = 16               # ile zapytań naraz
CHUNK_SIZE = 500               # co tyle ogłoszeń zapisujemy wyniki do pliku
OUTPUT_PATH = "llm_features.csv"
RUN_FULL = True               # zmień na True, żeby uruchomić (ok. $1, kilka-kilkanaście minut)

done_ids = set(pd.read_csv(OUTPUT_PATH)["id"]) if os.path.exists(OUTPUT_PATH) else set()
todo = df[~df["id"].isin(done_ids)]
print(f"Gotowe: {len(done_ids)}, do zrobienia: {len(todo)} ogłoszeń")
if metrics.get(SELECTED_MODEL) and metrics[SELECTED_MODEL].calls:
    per_ad = metrics[SELECTED_MODEL].total_cost_usd / len(metrics[SELECTED_MODEL].calls)
    print(f"Szacowany koszt (na podstawie kroku 5): ~${per_ad * len(todo):.2f}")

Gotowe: 0, do zrobienia: 5954 ogłoszeń
Szacowany koszt (na podstawie kroku 5): ~$1.10


In [42]:
async_client = AsyncOpenAI(api_key=api_key, max_retries=5)
semaphore = asyncio.Semaphore(CONCURRENCY)
full_metrics = OpenAIMetricsSession()


async def extract_row(ad_id, description) -> Optional[dict]:
    async with semaphore:  # czekamy, aż zwolni się miejsce (maks. CONCURRENCY zapytań naraz)
        try:
            response = await achat_completions_parse_with_metrics(
                async_client,
                session=full_metrics,
                model=SELECTED_MODEL,
                messages=build_messages(description),
                response_format=ApartmentFeatures,
                reasoning_effort="none",
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            return {"id": ad_id, **response.choices[0].message.parsed.model_dump()}
        except Exception as e:  # jedno problematyczne ogłoszenie nie powinno zatrzymać całości
            print(f"id={ad_id} błąd: {e}")
            return None


if RUN_FULL:
    t0 = time.perf_counter()
    for start in range(0, len(todo), CHUNK_SIZE):
        chunk = todo.iloc[start:start + CHUNK_SIZE]
        rows = await asyncio.gather(*(extract_row(i, d) for i, d in zip(chunk["id"], chunk["description"])))
        rows = [r for r in rows if r is not None]
        if rows:
            pd.DataFrame(rows).to_csv(OUTPUT_PATH, mode="a", header=not os.path.exists(OUTPUT_PATH), index=False)
        print(f"{start + len(chunk)}/{len(todo)} ogłoszeń, {time.perf_counter() - t0:.0f} s")
    print(f"\nCzas całości: {time.perf_counter() - t0:.0f} s "
          f"(suma czasów pojedynczych zapytań: {full_metrics.total_elapsed_s:.0f} s)")
    print(f"Koszt: ${full_metrics.total_cost_usd or 0:.2f}")
else:
    print("RUN_FULL = False - nic nie zostało wysłane.")

500/5954 ogłoszeń, 66 s
1000/5954 ogłoszeń, 129 s
1500/5954 ogłoszeń, 193 s
2000/5954 ogłoszeń, 256 s
2500/5954 ogłoszeń, 320 s
3000/5954 ogłoszeń, 387 s
3500/5954 ogłoszeń, 452 s
4000/5954 ogłoszeń, 517 s
4500/5954 ogłoszeń, 581 s
5000/5954 ogłoszeń, 645 s
5500/5954 ogłoszeń, 708 s
5954/5954 ogłoszeń, 765 s

Czas całości: 765 s (suma czasów pojedynczych zapytań: 12007 s)
Koszt: $1.08


### Połączenie z podziałem train/test
Każdy wiersz ma teraz informację, czy należy do zbioru treningowego, czy testowego.

In [43]:
SPLIT_PATH = "train_test_split.csv"
split = pd.DataFrame({"id": df["id"], "split": "df_train"})
split.loc[df_test.index, "split"] = "df_test"
split.to_csv(SPLIT_PATH, index=False)

In [44]:
if os.path.exists(OUTPUT_PATH):
    llm_features = pd.read_csv(OUTPUT_PATH).merge(split, on="id")
    print(llm_features["split"].value_counts())
    display(llm_features.head())

split
df_train    4758
df_test     1196
Name: count, dtype: int64


,id,neighborhood_name,near_public_transport,city_center_travel_time,nearby_green_areas,renovation_recency,finish_standard,layout_functionality,balcony_area_sqm,natural_light_exposure,num_bathrooms,parking,has_elevator,has_storage,split
0,5,Rondo Rataje,True,NaN,Przestronne tereny rekreacyjne nad Wartą; ziel...,NaN,NaN,open_kitchen,NaN,Okna z wystawą na wschód.,1.0,street,NaN,True,df_train
1,3525,"Rataje, os. Piastowskie",True,NaN,Tereny nad Wartą,NaN,NaN,walk_through,NaN,Bardzo jasne mieszkanie; okna z wystawą na zachód,1.0,NaN,NaN,True,df_train
2,4929,NaN,True,NaN,Tereny nadwarciańskie,Świeżo po remoncie,NaN,separate,NaN,Jasne mieszkanie; widok na wschodnią część miasta,NaN,NaN,NaN,NaN,df_train
3,7453,NaN,NaN,NaN,"Tereny nadwarciańskie, liczna zieleń, staw",NaN,shell_and_core,open_kitchen,14.0,NaN,NaN,garage,True,True,df_train
4,7573,Stare Żegrze,NaN,NaN,Kompleks rekreacyjno-sportowy z Jeziorem Malta...,NaN,NaN,NaN,NaN,NaN,NaN,garage,True,NaN,df_train


In [45]:
df_main = plik
df_llm = pd.read_csv("llm_features.csv")

# Połączenie tabel po kolumnie 'id'
merged_df = pd.merge(df_main, df_llm, on="id", how="inner")

print(f"Liczba wierszy po połączeniu: {merged_df.shape[0]}")
print(f"Liczba kolumn po połączeniu: {merged_df.shape[1]}")
display(merged_df.head(3))

Liczba wierszy po połączeniu: 5954
Liczba kolumn po połączeniu: 28


,id,market,created_at_first,created_at,district_lon,district_lat,title,description,params,price,...,nearby_green_areas,renovation_recency,finish_standard,layout_functionality,balcony_area_sqm,natural_light_exposure,num_bathrooms,parking,has_elevator,has_storage
0,49341,secondary,2018-01-04 13:11:05,2018-01-08 11:27:35,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,...,NaN,NaN,NaN,separate,NaN,"Jasna, oddzielna kuchnia.",1.0,NaN,NaN,NaN
1,261720,secondary,2018-01-15 19:12:07,2018-01-25 11:11:31,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>379000<br>price[curre...,379000.0,...,Tereny spacerowe i rekreacyjne; w pobliżu pows...,NaN,NaN,open_kitchen,NaN,NaN,1.0,NaN,NaN,True
2,50441,secondary,2018-01-04 13:53:29,2018-01-11 05:55:12,16.9408,52.38356,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,...,Blok otoczony zielenią; na osiedlu park i plac...,Balkon remontowany w ubiegłym roku; mieszkanie...,NaN,separate,NaN,Okna skierowane na zachód i wschód; mieszkanie...,1.0,NaN,NaN,True


In [46]:
merged_df.to_csv("real_estate_with_llm_features.csv", index=False)